# 论文复现: TeleAI at SemEval-2026 Task 3 — Subtask 1 (DimASR, VA 回归)

> 复现论文 TeleAI, Xingchen AGI Lab的完整系统。
>
> 给定文本 + Aspect，预测连续 Valence–Arousal 分数，官方指标 RMSE_VA。



论文未采用的部分: 多阶段领域迁移训练、负 Pearson 相关损失（无一致增益），
以及 Subtask 2/3 的直接 JSON 提示。

## 0. 环境配置

In [ ]:
!pip install -q peft transformers accelerate bitsandbytes scipy

In [ ]:
import json
import math
import random
import re
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from scipy.stats import pearsonr

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import sys
import peft.import_utils
import peft.tuners.lora.model      # noqa: F401  确保相关子模块已被导入
import peft.tuners.lora.torchao    # noqa: F401

peft.import_utils.is_torchao_available = lambda: False
for _m in list(sys.modules.values()):
    if _m is None:
        continue
    _f = getattr(_m, 'is_torchao_available', None)
    if _f is not None and getattr(_f, '__module__', '').startswith('peft'):
        _m.is_torchao_available = lambda: False
print('已禁用 peft 的 torchao 版本检查')

## 一、超参数配置

In [ ]:
_CC = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
COMPUTE_DTYPE = torch.bfloat16 if _CC[0] >= 8 else torch.float16
print(f"GPU capability: {_CC} | compute dtype: {COMPUTE_DTYPE}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

CONFIG = {
    'val_ratio': 0.10,           # 每个训练文件切 10% 作本地 val
    'train_langs': ['zho'],      # T4×2 预算: 仅 zho 三域混合; A100 全量复现设 None（全部多语言混合）
    'max_len': 128,              # 论文: max_len=128, right padding
    'model_name': 'Qwen/Qwen2.5-1.5B', 
    'use_4bit': False,        
    'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'use_adapter_head': False,   # False = 全量 mean-pooling 头
    'epochs': 10, 'per_gpu_bs': 4, 'grad_accum': 1,   
    'lr_head': 1e-3, 'lr_lora': 1e-4,
    'weight_decay': 0.01, 'warmup_ratio': 0.1, 'grad_clip': 1.0,
    'seed': 42,
    'huber_beta': 0.5,
    'rdrop_alpha': 0.5,
    'pgd_eps': 0.02, 'pgd_steps': 3, 'pgd_lambda': 0.5,
    'use_lora': True,             # False = 全参数 SFT (Baseline)
    'use_huber': True,            # False = 标准 MSE
    'use_rdrop': True,            # False = 去 R-Drop 一致性
    'use_pgd': True,              # False = 去 PGD 对抗训练
    'use_interval_map': True,     # False = 去 sigmoid 区间映射（直接 clip）
    'use_diff_lr_warmup': True,   # False = 统一 lr、无 warmup/衰减
    'use_calibration': True,      # False = 去后验线性校准
}

set_seed(CONFIG['seed'])
print("配置完成（对应论文 Table 2）")

## 二、数据路径

In [ ]:
def find_data_root():
    candidates = [
        Path('DimABSA2026/task-dataset/track_a/subtask_1'),
        Path('/Users/piaoxiang/Documents/DimABSA/DimABSA2026/task-dataset/track_a/subtask_1'),
        Path('/kaggle/input'),
        Path('/content'),
    ]
    for root in candidates:
        if not root.exists():
            continue
        hits = list(root.rglob('*_train_*.jsonl'))
        track_a = [p for p in hits if 'track_a' in str(p)]
        hits = track_a or hits
        if hits:
            return hits[0].parent.parent if len(hits) == 1 else root
    raise FileNotFoundError('找不到 *_train_*.jsonl, 请手动指定 DATA_ROOT')

DATA_ROOT = find_data_root()
TRAIN_FILES = sorted(DATA_ROOT.rglob('*_train_*.jsonl'))
DEV_FILES = sorted(DATA_ROOT.rglob('*_dev_task1.jsonl'))

if CONFIG['train_langs'] is not None:
    TRAIN_FILES = [p for p in TRAIN_FILES if p.name.split('_')[0] in CONFIG['train_langs']]
    DEV_FILES = [p for p in DEV_FILES if p.name.split('_')[0] in CONFIG['train_langs']]   # dev 同步过滤为 zho

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"训练文件 ({len(TRAIN_FILES)}):")
for p in TRAIN_FILES:
    print(f"  {p.name}")
print(f"dev 文件 ({len(DEV_FILES)}): {[p.name for p in DEV_FILES]}")

## 三、数据解析与实例展开

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

VA_KEYS = ['Aspect_VA', 'Triplet', 'Quadruplet']   # 不同结构的 aspect-VA 载体字段

def explode_instances(data):
    """展平为一行一个 aspect, 对 (ID, Aspect) 去重保留首个"""
    rows, seen = [], set()
    for item in data:
        for key in VA_KEYS:
            entries = item.get(key)
            if not entries:
                continue
            for e in entries:
                aspect = e.get('Aspect', '')
                try:
                    v, a = map(float, e['VA'].split('#'))
                except Exception:
                    continue
                if (item['ID'], aspect) in seen:
                    continue
                seen.add((item['ID'], aspect))
                rows.append({'ID': item['ID'], 'Text': item['Text'], 'Aspect': aspect,
                             'V': v, 'A': a})
            break   # 同一条样本只取第一个出现的结构
    return rows

print("解析工具定义完成")

## 四、多语言 / 多领域混合训练集

In [ ]:
def file_lang_domain(path):
    m = re.match(r'^(?P<lang>[a-z]{2,3})_(?P<domain>.+?)_train', path.name)
    if m:
        return m['lang'], m['domain']
    return 'unk', path.stem

train_rows, val_rows = [], []
for fi, fp in enumerate(TRAIN_FILES):
    lang, domain = file_lang_domain(fp)
    rows = explode_instances(load_jsonl(fp))
    random.Random(CONFIG['seed'] + fi).shuffle(rows)
    n_val = max(1, int(len(rows) * CONFIG['val_ratio']))
    for r in rows[:n_val]:
        r['source'] = f'{lang}_{domain}'
    for r in rows[n_val:]:
        r['source'] = f'{lang}_{domain}'
    val_rows += rows[:n_val]
    train_rows += rows[n_val:]

train_df = pd.DataFrame(train_rows)
val_df = pd.DataFrame(val_rows)

print(f"混合训练集: {len(train_df):,} 条展开样本 | 本地 val: {len(val_df):,} 条")
print(f"语言-领域分布 (train):\n{train_df.groupby('source').size().to_string()}")
print(f"\nVA 标签统计 (train):")
print(train_df[['V', 'A']].describe().loc[['min', 'max', 'mean', 'std']].round(2).to_string())

## 五、输入模板与数据集

In [ ]:
def format_input(aspect, text):
    return f"[ASPECT] {aspect}[/ASPECT] \n[TEXT] {text}[/TEXT]"

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'          # 论文: right padding

class VaDataset(Dataset):
    """预分词的数据集: (input_ids, attention_mask, labels=[V, A])"""
    def __init__(self, df, max_len):
        enc = tokenizer(
            [format_input(r.Aspect, r.Text) for r in df.itertuples()],
            truncation=True, max_length=max_len, padding='max_length',
            return_tensors='pt', add_special_tokens=False,
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        # 任务规定 VA ∈ [1,9]; 个别原始标签越界, 训练标签按区间裁剪
        self.labels = torch.tensor(df[['V', 'A']].to_numpy(dtype=np.float64).clip(1.0, 9.0), dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        return {'input_ids': self.input_ids[i],
                'attention_mask': self.attention_mask[i],
                'labels': self.labels[i]}

print("模板与数据集定义完成")

## 六、模型: Qwen2.5-7B + LoRA + mean pooling + sigmoid 区间映射（

In [ ]:
class VaRegressor(nn.Module):
    """主干(Qwen2.5+LoRA) -> mask-aware mean pooling -> 线性头 -> [1,9] 区间映射"""

    def __init__(self, backbone, hidden_size, cfg):
        super().__init__()
        self.backbone = backbone
        self.cfg = cfg
        self.head = nn.Linear(hidden_size, 2)
        nn.init.zeros_(self.head.bias)
        if cfg['use_adapter_head']:
            self.adapter = nn.Sequential(nn.Linear(hidden_size, 512), nn.GELU(), nn.Linear(512, 2))

    def get_input_embeddings(self):
        return self.backbone.get_input_embeddings()

    def mean_pool(self, hidden, mask):
        m = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * m).sum(1) / (m.sum(1) + 1e-6)

    def forward(self, input_ids, attention_mask, delta=None, emb0=None):
        if emb0 is None:
            emb0 = self.get_input_embeddings()(input_ids)
        inputs_embeds = (emb0.float() + delta).to(emb0.dtype) if delta is not None else emb0
        out = self.backbone(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        h = self.mean_pool(out.last_hidden_state, attention_mask).float()   # 最终层隐状态, 转 fp32
        h = h.to(self.head.weight.device)   
        z = self.adapter(h) if self.cfg['use_adapter_head'] else self.head(h)
        if self.cfg['use_interval_map']:
            return 1.0 + 8.0 * torch.sigmoid(z)          
        return z.clamp(1.0, 9.0)                          

def build_model(cfg):
    torch_dtype = COMPUTE_DTYPE
    if cfg['use_4bit']:
        backbone = AutoModel.from_pretrained(
            cfg['model_name'], trust_remote_code=True, device_map='auto',
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch_dtype),
        )
    else:
        backbone = AutoModel.from_pretrained(
            cfg['model_name'], trust_remote_code=True, torch_dtype=torch_dtype).to(DEVICE)

    backbone.config.use_cache = False
    backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

    if cfg['use_lora']:
        lora = LoraConfig(r=cfg['lora_r'], lora_alpha=cfg['lora_alpha'],
                          lora_dropout=cfg['lora_dropout'],
                          target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                          bias='none')
        backbone = PeftModel(backbone, lora)
    else:
        if cfg['use_4bit']:
            print('[提示] 全参数微调与 4-bit 量化冲突, 自动关闭 4bit')
            return build_model({**cfg, 'use_4bit': False})
        for p in backbone.parameters():
            p.requires_grad = True

    model = VaRegressor(backbone, backbone.config.hidden_size, cfg)
    model.head = model.head.to(DEVICE)
    if cfg['use_adapter_head']:
        model.adapter = model.adapter.to(DEVICE)
    return model

print("模型定义完成")

## 七、Huber + R-Drop + embedding

In [ ]:
def get_criterion(cfg):
    return (nn.SmoothL1Loss(beta=cfg['huber_beta']) if cfg['use_huber']
            else nn.MSELoss())                     

def pgd_attack(model, input_ids, attention_mask, labels, cfg):
    with torch.no_grad():
        emb0 = model.get_input_embeddings()(input_ids)
    delta = torch.zeros(emb0.shape, dtype=torch.float32, device=emb0.device).requires_grad_(True)
    crit = get_criterion(cfg)
    eps, eta = cfg['pgd_eps'], cfg['pgd_eps'] / cfg['pgd_steps']
    for _ in range(cfg['pgd_steps']):
        loss = crit(model(input_ids, attention_mask, delta=delta, emb0=emb0), labels)
        g = torch.autograd.grad(loss, delta)[0]
        delta = (delta.detach() + eta * g.sign()).clamp(-eps, eps).detach().requires_grad_(True)
    return delta.detach()

def training_step(model, batch, cfg):
    ids = batch['input_ids'].to(DEVICE)
    am = batch['attention_mask'].to(DEVICE)
    y = batch['labels'].to(DEVICE)
    crit = get_criterion(cfg)

    # 1. R-Drop 双前向
    p1 = model(ids, am)
    if cfg['use_rdrop']:
        p2 = model(ids, am)
        L_clean = 0.5 * (crit(p1, y) + crit(p2, y)) + cfg['rdrop_alpha'] * F.mse_loss(p1, p2)
    else:
        L_clean = crit(p1, y)

    # 2-3. PGD 对抗步 + 总损失
    if cfg['use_pgd']:
        delta = pgd_attack(model, ids, am, y, cfg)
        L_adv = crit(model(ids, am, delta=delta, emb0=None), y)
        return L_clean + cfg['pgd_lambda'] * L_adv
    return L_clean

print("损失与训练步骤定义完成")

## 八、优化器: 差分学习率 + warmup + 线性衰减

head `lr=1e-3`、LoRA `lr=1e-4`、AdamW wd=0.01、warmup_ratio=0.1 后线性衰减、梯度裁剪 1.0。

In [ ]:
def build_optimizer_scheduler(model, cfg, total_steps):
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith(('head.', 'adapter.'))]
    lora_params = [p for n, p in model.named_parameters() if p.requires_grad and 'lora_' in n]
    base_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith(('head.', 'adapter.')) and 'lora_' not in n]

    if cfg['use_diff_lr_warmup']:
        groups = [{'params': head_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
                  {'params': lora_params, 'lr': cfg['lr_lora'], 'weight_decay': cfg['weight_decay']},
                  {'params': base_params, 'lr': cfg['lr_lora'], 'weight_decay': cfg['weight_decay']}]
    else:  
        groups = [{'params': head_params + lora_params + base_params, 'lr': 1e-4, 'weight_decay': 0.0}]
    optimizer = torch.optim.AdamW(groups)

    if cfg['use_diff_lr_warmup']:
        warmup = max(1, int(total_steps * cfg['warmup_ratio']))
        def lr_lambda(step):
            if step < warmup:
                return step / warmup
            return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    else:
        scheduler = None
    return optimizer, scheduler

print("优化器/调度器定义完成")

## 九、评估: RMSE_VA 与 PCC

In [ ]:
@torch.no_grad()
def predict_va(model, df, batch_size=16):
    model.eval()
    ds = VaDataset(df, CONFIG['max_len'])
    loader = DataLoader(ds, batch_size=batch_size)
    preds = []
    for batch in tqdm(loader, desc='预测', leave=False):
        p = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        preds.append(p.float().cpu().numpy())
    return np.concatenate(preds)          

def rmse_va(gold, pred, do_norm=True):
    diff2 = np.sum((gold - pred) ** 2, axis=1)
    val = math.sqrt(diff2.sum() / len(gold))
    return val / math.sqrt(128) if do_norm else val

def evaluate_va(gold, pred):
    return {'RMSE_VA': rmse_va(gold, pred), 'RMSE_VA_raw': rmse_va(gold, pred, do_norm=False),
            'PCC_V': pearsonr(pred[:, 0], gold[:, 0])[0],
            'PCC_A': pearsonr(pred[:, 1], gold[:, 1])[0]}

print("评估函数定义完成")

## 十、训练主循环

In [ ]:
def snapshot_trainable(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def restore_trainable(model, state):
    for n, p in model.named_parameters():
        if n in state:
            p.data.copy_(state[n].to(p.device))

def train_model(cfg, train_df, val_df):
    set_seed(cfg['seed'])
    train_loader = DataLoader(VaDataset(train_df, cfg['max_len']),
                              batch_size=cfg['per_gpu_bs'], shuffle=True)
    total_steps = math.ceil(len(train_loader) / cfg['grad_accum']) * cfg['epochs']

    model = build_model(cfg)
    optimizer, scheduler = build_optimizer_scheduler(model, cfg, total_steps)
    gold_val = val_df[['V', 'A']].to_numpy()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

    history, best_rmse, best_state = [], float('inf'), None
    for epoch in range(cfg['epochs']):
        model.train()
        epoch_loss, opt_step = 0.0, 0
        optimizer.zero_grad()
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{cfg["epochs"]}', leave=False):
            loss = training_step(model, batch, cfg) / cfg['grad_accum']
            loss.backward()
            epoch_loss += loss.item() * cfg['grad_accum']
            if (opt_step + 1) % cfg['grad_accum'] == 0:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['grad_clip'])
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                optimizer.zero_grad()
            opt_step += 1

        preds = predict_va(model, val_df)
        val_rmse = rmse_va(gold_val, preds)
        history.append({'epoch': epoch + 1, 'train_loss': epoch_loss / len(train_loader), 'val_rmse_va': val_rmse})
        print(f"Epoch {epoch+1}: loss={history[-1]['train_loss']:.4f} | val RMSE_VA={val_rmse:.4f}", end='')
        if val_rmse < best_rmse:
            best_rmse, best_state = val_rmse, snapshot_trainable(model)
            print('  ✓ 最佳')
        else:
            print()

    if best_state is not None:
        restore_trainable(model, best_state)
    print(f"最佳 val RMSE_VA: {best_rmse:.4f}")
    return model, pd.DataFrame(history), best_rmse

print("训练流程定义完成")

## 十一、后验线性校准

In [ ]:
def fit_calibration(preds, gold):
    """对 V/A 分别做单变量线性回归"""
    return [tuple(np.polyfit(preds[:, j], gold[:, j], 1)) for j in range(2)]

def apply_calibration(preds, coefs):
    out = np.stack([a * preds[:, j] + b for j, (a, b) in enumerate(coefs)], axis=1)
    return np.clip(out, 1.0, 9.0)

print("校准函数定义完成")

## 十二、正式训练 

In [ ]:
def save_submission(df, pred, out_path):
    grouped = OrderedDict()
    for (i, r), (v, a) in zip(df.iterrows(), pred):
        grouped.setdefault(r['ID'], []).append(
            {'Aspect': r['Aspect'], 'VA': f'{v:.2f}#{a:.2f}'})
    with open(out_path, 'w', encoding='utf-8') as f:
        for id_, avs in grouped.items():
            text = df.loc[df['ID'] == id_, 'Text'].iloc[0]
            f.write(json.dumps({'ID': id_, 'Text': text, 'Aspect_VA': avs}, ensure_ascii=False) + '\n')

def evaluate_on_files(model, files, coefs=None, save_sub=False):
    out = []
    for fp in files:
        df = pd.DataFrame(explode_instances(load_jsonl(fp)))
        preds = predict_va(model, df)
        gold = df[['V', 'A']].to_numpy()
        row = {'file': fp.name, **evaluate_va(gold, preds)}
        if coefs is not None:
            row.update({f'{k}_calib': v for k, v in evaluate_va(gold, apply_calibration(preds, coefs)).items()})
        out.append(row)
        if save_sub:
            p = apply_calibration(preds, coefs) if coefs is not None else preds
            save_submission(df, p, PROJECT_ROOT / f'submission_{fp.stem}.jsonl')
    return pd.DataFrame(out)

PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

print("评测与提交流程定义完成")

In [ ]:
# ===== 正式运行最终系统 =====
model, history, best_val_rmse = train_model(CONFIG, train_df, val_df)

# 后验校准: 在 val 上拟合 (§3.8)
calib_coefs = None
if CONFIG['use_calibration']:
    val_preds = predict_va(model, val_df)
    calib_coefs = fit_calibration(val_preds, val_df[['V', 'A']].to_numpy())
    print(f"校准系数 (V/A): {[(round(a,3), round(b,3)) for a, b in calib_coefs]}")

dev_results = evaluate_on_files(model, DEV_FILES, coefs=calib_coefs, save_sub=True)
dev_results

## 十三、消融实验

In [ ]:
ABLATIONS = [
    ('Our Final System (LoRA + All Components)', {}),
    ('Replace LoRA with Full SFT', {'use_lora': False, 'use_4bit': False}),  
    ('w/o PGD Adversarial Training', {'use_pgd': False}),
    ('w/o R-Drop Consistency', {'use_rdrop': False}),
    ('w/o Output Interval Mapping', {'use_interval_map': False}),
    ('w/o Huber Loss (using standard MSE)', {'use_huber': False}),
    ('w/o Differential LR & Warmup/Decay', {'use_diff_lr_warmup': False}),
    ('w/o Post-hoc Linear Calibration', {'use_calibration': False}),
]

# Table 1 (dev, English-Restaurant): 1.03 / 0.85 / 0.91 / 0.94 / 0.92 / 0.89 / 0.88 / 0.87 / 0.87
ABLATION_EPOCHS = 1
ABLATION_SUBSET = [0, 2, 3]    # None = 全部 8 组
abl_rows = []
for ai, (name, overrides) in enumerate(ABLATIONS):
    if ABLATION_SUBSET is not None and ai not in ABLATION_SUBSET:
        continue
    cfg = dict(CONFIG, **overrides)
    if ABLATION_EPOCHS:
        cfg['epochs'] = ABLATION_EPOCHS
    print(f"\n{'#'*60}\n消融: {name}\n{'#'*60}")
    m, _, val_rmse = train_model(cfg, train_df, val_df)
    coefs = None
    if cfg['use_calibration']:
        vp = predict_va(m, val_df)
        coefs = fit_calibration(vp, val_df[['V', 'A']].to_numpy())
    dr = evaluate_on_files(m, DEV_FILES, coefs=coefs)
    eng_rest = dr[dr['file'].str.startswith('eng')]
    abl_rows.append({'ablation': name, 'val_rmse_va': round(val_rmse, 4),
                     'dev_rmse_va_calib': round(float(dr['RMSE_VA_calib'].mean()) if 'RMSE_VA_calib' in dr else float(dr['RMSE_VA'].mean()), 4)})
    del m
    torch.cuda.empty_cache()

ablation_df = pd.DataFrame(abl_rows)
ablation_df.to_csv(PROJECT_ROOT / 'paper_repro_ablation.csv', index=False, encoding='utf-8-sig')
ablation_df

## 十四、可视化

In [ ]:
import matplotlib.pyplot as plt

ACL_COLORS = {'blue': '#3175B0', 'orange': '#E69C2E', 'red': '#CC3333', 'green': '#4C8C2B'}
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['epoch'], history['val_rmse_va'], 'o-', color=ACL_COLORS['blue'])
ax1.set_xlabel('Epoch'); ax1.set_ylabel('val RMSE_VA')
ax1.set_title('Training curve (val RMSE_VA, best-ckpt selection)')

if len(ablation_df):
    short = [a.replace('w/o ', 'w/o\n') for a in ablation_df['ablation']]
    ax2.bar(range(len(ablation_df)), ablation_df['dev_rmse_va_calib'], color=ACL_COLORS['orange'], alpha=0.85)
    for i, v in enumerate(ablation_df['dev_rmse_va_calib']):
        ax2.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
    ax2.set_xticks(range(len(short))); ax2.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
    ax2.set_ylabel('dev RMSE_VA')
    ax2.set_title('Ablation (cf. paper Table 1: final 0.85)')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'paper_repro_results.png', dpi=300, bbox_inches='tight')
plt.show()